In [2]:
import os
import google.generativeai as genai
import requests
import re
import json
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
import faiss

class ProoomptEngineer:
    key = os.environ.get("GEMINI_API_KEY")
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-pro:generateContent?key={key}"
    headers = { 'Content-Type': 'application/json' }

    def _get_llm_response(self, system_prooompt: str, prompt: str) -> str:

        data = {
            "system_instruction": {
                "parts": [
                    {"text": system_prooompt}
                ]
            },
            "contents": {
                "parts": [
                    {"text": prompt}
                ]
            }
        }

        try:
            response = requests.post(self.url, headers=self.headers, data=json.dumps(data))
            response.raise_for_status()
            response_json = response.json()
            text = response_json["candidates"][0]["content"]["parts"][0]["text"]
            return text
        except requests.exceptions.RequestException as e:
            return ""

    def _non_Discord_Code_Block_ificator(self, json_response: str) -> str:
        json_response = re.sub("```$", "", json_response)
        json_response = re.sub("^```json", "", json_response)

        return json_response

    def try_get_name(self, speechToText: str) -> tuple:
        namesystemprompt = """
            Your goal is to take a user input and extract a name if they told you.
            If you didnt find a name you should set name to null.
            If there are mutliple names, set the name to null
            Only reply with raw json in the format of:
            {
                "response": {
                    "name": "extracted name"
                }
            }
        """

        response = self._get_llm_response(namesystemprompt, speechToText)

        if len(response.strip()) == 0:
            return False, None

        response = self._non_Discord_Code_Block_ificator(response)

        name = json.loads(response)['response']['name']

        return (True, name)

# Example Usage
if __name__ == "__main__":
    instruction_file = 'src/systemprompt.txt'
    system_prooompt = ''
    try:
        with open(instruction_file, 'r') as file:
            system_prooompt = file.read().strip()
    except FileNotFoundError:
        system_prooompt = "Provide helpful and concise responses."

    rokosBasilisk = ProoomptEngineer()

    # rokosBasilisk.set_system_prompt(rokosBasilisk, system_prooompt)
    #response = rokosBasilisk._get_llm_response(system_prooompt, "hello")

    #print(response)


C:\Users\trist\PycharmProjects\RAG-Studienarbeit\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
SRC_DIR = "./src/"

Create a Vector index:


In [4]:
import os


# ---------------------------
# Step 1: Load Processed Data
# ---------------------------
def load_chunks(json_path):
    """
    Load processed text chunks from a JSON file.
    Expected JSON format: a list of dictionaries with at least a "text" key.
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    # Extract texts; adjust if your JSON structure differs
    texts = [chunk["text"] for chunk in data]
    return texts, data  # also returning the full data for metadata if needed

# ----------------------------------------
# Step 2: Generate Embeddings with Contriever
# ----------------------------------------
def compute_embeddings(texts, tokenizer, model, device="cpu", batch_size=16):
    """
    Compute embeddings for a list of texts using Contriever.
    """
    model.to(device)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i : i + batch_size]
            # Tokenize the batch. Adjust max_length if needed.
            inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=256, return_tensors="pt")
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs)
            # The Contriever model returns the embeddings in last_hidden_state[:,0,:] or use a pooling strategy.
            # Here, we assume a pooling strategy is implemented (or use mean pooling).
            # For demonstration, we use mean pooling on the token embeddings (excluding padding tokens)
            token_embeddings = outputs.last_hidden_state  # shape: (batch_size, seq_len, hidden_dim)
            attention_mask = inputs["attention_mask"].unsqueeze(-1)  # shape: (batch_size, seq_len, 1)
            # Avoid division by zero; sum and average over non-padding tokens
            sum_embeddings = torch.sum(token_embeddings * attention_mask, dim=1)
            valid_tokens = torch.clamp(attention_mask.sum(dim=1), min=1e-9)
            batch_embeddings = sum_embeddings / valid_tokens
            embeddings.append(batch_embeddings.cpu().numpy())
    embeddings = np.vstack(embeddings)
    return embeddings

# ----------------------------------
# Step 3: Build a FAISS Vector Index
# ----------------------------------
def build_faiss_index(embeddings):
    """
    Build a FAISS index (using L2 distance) from embeddings.
    """
    d = embeddings.shape[1]  # dimension of embeddings
    index = faiss.IndexFlatL2(d)
    index.add(embeddings.astype('float32'))
    return index

# ---------------------------
# Main Execution
# ---------------------------
if __name__ == "__main__":
    # Path to your JSON file with processed chunks
    json_path = SRC_DIR + "processed_chunks.json"  # <-- change to your file path

    # Load texts and metadata
    texts, metadata = load_chunks(json_path)
    print(f"Loaded {len(texts)} chunks.")

    # Initialize Contriever model and tokenizer.
    # The model name "facebook/contriever" is used here; adjust if you have a different checkpoint.
    model_name = "facebook/contriever"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    # Compute embeddings for the texts
    device = "cuda" if torch.cuda.is_available() else "cpu"
    embeddings = compute_embeddings(texts, tokenizer, model, device=device)
    print(f"Computed embeddings with shape: {embeddings.shape}")

    # Build a FAISS index
    index = build_faiss_index(embeddings)
    print("FAISS index built.")
    # Create the directory if it doesn't exist
    os.makedirs("./indices", exist_ok=True)

    # Assume 'index' is your FAISS index
    faiss.write_index(index, "./indices/my_faiss_index.idx")

    # Example: Search the index with a query embedding (optional)
    # query = "Your sample query here"
    # query_embedding = compute_embeddings([query], tokenizer, model, device=device)
    # distances, indices = index.search(query_embedding.astype('float32'), k=5)
    # print("Top-5 results:", indices)


Loaded 44 chunks.
Computed embeddings with shape: (44, 768)
FAISS index built.


Faiss Tests

In [ ]:
# Example: Search the index with a query embedding (optional)
index = faiss.read_index("./indices/my_faiss_index.idx")

query = "Whats the difference between DIVERSA premium and DIVERSA care in Outpatient treatments for orthodox medicine?"
query_embedding = compute_embeddings([query], tokenizer, model, device=device)
distances, indices = index.search(query_embedding.astype('float32'), k=5)
print("Top-5 results:", indices)

In [4]:
import os


Markitdown tests


In [5]:
from markitdown import MarkItDown

md = MarkItDown()
result = md.convert("Docs/sample-1.pptx")
print(result.text_content)

<!-- Slide number: 1 -->
# Discoverability and Analytics

### Notes:

<!-- Slide number: 2 -->
# Progressive Enhancement
Discoverability and PWA’s

Progressive enhancement = content available to all users
JS sites are indexed by Google**
Test your site with Fetch as Google tool

**Follow these best practices

### Notes:
Ref: https://plus.google.com/+JohnMueller/posts/LT4fU7kFB8W

Progressive Enhancement
Use "feature detection" & "progressive enhancement"  techniques to make your content available to all users.
Avoid redirecting to an "unsupported browser" page.
Consider using a polyfill or other safe fallback where needed.

JS based sites
Google crawler needs to index what’s going on in your site.
People often wonder if the crawler will be able to do this for JS-based sites.
Google’s crawler will work with JS, but there are some specific practices that have to be followed - these are detailed in this post.

Test your site
There is a tool called “Fetch as Google” that enables you to tes